In [14]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# ===== 0. 模型定義 =====
class F1Net(nn.Module):
    def __init__(self, cat_dims, num_num, emb_dim=8, hidden_dim=64):
        super().__init__()
        self.emb_layers = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.mlp = nn.Sequential(
            nn.Linear(len(cat_dims)*emb_dim + num_num, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x_cat, x_num):
        embs = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x).squeeze(-1)

# ===== 1. 讀取資料 =====
winners = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')
laps = pd.read_csv('./data/fastest_laps_updated.csv')

# ===== 2. 日期與欄位預處理 =====
winners['Date'] = pd.to_datetime(winners['Date'])
winners['year'] = winners['Date'].dt.year
winners['race_order'] = winners.groupby('year')['Date'].rank(method='first').astype(int)

laps['year'] = laps['year'].astype(int)
laps = pd.merge(
    laps,
    winners[['Grand Prix', 'year', 'Date']].drop_duplicates(),
    on=['Grand Prix', 'year'],
    how='left'
)

def time_to_seconds(tstr):
    if pd.isnull(tstr) or tstr == '':
        return np.nan
    parts = str(tstr).split(':')
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    else:
        return np.nan
laps['Time_sec'] = laps['Time'].apply(time_to_seconds)

# ===== 3. 特徵與標籤生成 =====
def generate_features(winners, drivers, laps):
    season_points = defaultdict(lambda: defaultdict(int))
    season_wins = defaultdict(lambda: defaultdict(int))
    team_points = defaultdict(lambda: defaultdict(int))
    records = []

    for year in sorted(winners['year'].unique()):
        races = winners[winners['year'] == year].sort_values('Date')
        for _, row in races.iterrows():
            race = row['Grand Prix']
            date = row['Date']
            round_num = row['race_order']
            season_drivers = drivers[drivers['year'] == year]

            for _, drow in season_drivers.iterrows():
                driver = drow['Driver']
                team = drow['Car']
                nationality = drow['Nationality']

                points_so_far = season_points[year][driver]
                wins_so_far = season_wins[year][driver]
                team_pts_so_far = team_points[year][team]

                prior_track_wins = winners[
                    (winners['Winner'] == driver) &
                    (winners['Grand Prix'] == race) &
                    (winners['Date'] < date)
                ]
                career_track_wins = len(prior_track_wins)

                relevant_laps = laps[
                    (laps['Driver'] == driver) &
                    (laps['Grand Prix'] == race) &
                    (laps['Date'] < date)
                ]
                if len(relevant_laps) > 0:
                    best_lap = relevant_laps['Time_sec'].min()
                    is_new_on_track = 0
                else:
                    best_lap = laps['Time_sec'].mean()
                    is_new_on_track = 1

                is_winner = int((row['Winner'] == driver) and (row['Car'] == team))

                records.append({
                    'Grand Prix': race,
                    'year': year,
                    'race_order': round_num,
                    'Date': date,
                    'Driver': driver,
                    'Team': team,
                    'Nationality': nationality,
                    'SeasonWinsSoFar': wins_so_far,
                    'SeasonPointsSoFar': points_so_far,
                    'CareerTrackWins': career_track_wins,
                    'DriverTrackBestLap': best_lap,
                    'IsNewOnTrack': is_new_on_track,
                    'TeamSeasonPointsSoFar': team_pts_so_far,
                    'is_winner': is_winner
                })

                if row['Winner'] == driver:
                    season_points[year][driver] += 25
                    season_wins[year][driver] += 1
                    team_points[year][team] += 25
    return pd.DataFrame(records)

df = generate_features(winners, drivers, laps)

# ===== 4. 編碼與標準化 =====
cat_cols = ['Grand Prix', 'Driver', 'Team', 'Nationality']
num_cols = ['year', 'race_order', 'SeasonWinsSoFar', 'SeasonPointsSoFar',
            'CareerTrackWins', 'DriverTrackBestLap', 'IsNewOnTrack', 'TeamSeasonPointsSoFar']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ===== 5. Dataset 類別 =====
class F1Dataset(torch.utils.data.Dataset):
    def __init__(self, df, cat_cols, num_cols):
        self.X_cat = df[cat_cols].values.astype(np.int64)
        self.X_num = df[num_cols].values.astype(np.float32)
        self.y = df['is_winner'].values.astype(np.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return [self.X_cat[idx], self.X_num[idx], self.y[idx]]

# ===== 6. 時序動態訓練 =====
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

window_size_days = 365
validation_period_days = 90

df = df.sort_values('Date')
start_date = df['Date'].min()
end_date = df['Date'].max()

cat_dims = [df[col].max() + 1 for col in cat_cols]
    
# 新增：資料洩漏檢查
def check_temporal_leakage(train_df, val_df):
    # 訓練資料最大日期
    train_max_date = train_df['Date'].max()
    # 驗證資料最小日期
    val_min_date = val_df['Date'].min()
    if val_min_date <= train_max_date:
        raise ValueError(f"資料洩漏疑慮：驗證集開始日期 {val_min_date} 不應小於等於訓練集最大日期 {train_max_date}")
    else:
        print(f"時序驗證通過：訓練集最大日期 {train_max_date}, 驗證集最小日期 {val_min_date}")

while start_date + pd.Timedelta(days=window_size_days) < end_date:
    train_end = start_date + pd.Timedelta(days=window_size_days)
    val_start = train_end
    val_end = train_end + pd.Timedelta(days=validation_period_days)

    train_data = df[df['Date'] < train_end]
    val_data = df[(df['Date'] >= val_start) & (df['Date'] < val_end)]

    if len(train_data) == 0 or len(val_data) == 0:
        print(f"Skipped period {val_start.date()} to {val_end.date()} due to empty data.")
        start_date += pd.Timedelta(days=validation_period_days)
        continue

    # 新增：資料洩漏檢查
    check_temporal_leakage(train_data, val_data)

    # 以下為訓練與驗證流程
    trainset = F1Dataset(train_data, cat_cols, num_cols)
    valset = F1Dataset(val_data, cat_cols, num_cols)
    trainloader = DataLoader(trainset, batch_size=256, shuffle=True)
    valloader = DataLoader(valset, batch_size=256, shuffle=False)

    model = F1Net(cat_dims, len(num_cols))
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(5):
        model.train()
        for X_cat, X_num, y in trainloader:
            X_cat, X_num, y = X_cat.to(device), X_num.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X_cat, X_num)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_cat, X_num, y in valloader:
            X_cat, X_num = X_cat.to(device), X_num.to(device)
            logits = model(X_cat, X_num)
            pred = (torch.sigmoid(logits) > 0.5).cpu().numpy()
            y_true.extend(y.numpy())
            y_pred.extend(pred)

    if len(y_true) == 0 or len(y_pred) == 0:
        print(f"Skipped accuracy calculation for period {val_start.date()} to {val_end.date()} due to empty predictions.")
    else:
        print(f"Validation from {val_start.date()} to {val_end.date()} accuracy: {accuracy_score(y_true, y_pred):.3f}")

    start_date += pd.Timedelta(days=validation_period_days)



時序驗證通過：訓練集最大日期 1950-09-03 00:00:00, 驗證集最小日期 1951-05-27 00:00:00
Validation from 1951-05-13 to 1951-08-11 accuracy: 0.414
時序驗證通過：訓練集最大日期 1951-07-29 00:00:00, 驗證集最小日期 1951-09-16 00:00:00
Validation from 1951-08-11 to 1951-11-09 accuracy: 0.947
Skipped period 1951-11-09 to 1952-02-07 due to empty data.
Skipped period 1952-02-07 to 1952-05-07 due to empty data.
時序驗證通過：訓練集最大日期 1951-10-28 00:00:00, 驗證集最小日期 1952-05-18 00:00:00
Validation from 1952-05-07 to 1952-08-05 accuracy: 0.955
時序驗證通過：訓練集最大日期 1952-08-03 00:00:00, 驗證集最小日期 1952-08-17 00:00:00
Validation from 1952-08-05 to 1952-11-03 accuracy: 0.955
時序驗證通過：訓練集最大日期 1952-09-07 00:00:00, 驗證集最小日期 1953-01-18 00:00:00
Validation from 1952-11-03 to 1953-02-01 accuracy: 0.947
Skipped period 1953-02-01 to 1953-05-02 due to empty data.
時序驗證通過：訓練集最大日期 1953-01-18 00:00:00, 驗證集最小日期 1953-05-30 00:00:00
Validation from 1953-05-02 to 1953-07-31 accuracy: 0.947
時序驗證通過：訓練集最大日期 1953-07-18 00:00:00, 驗證集最小日期 1953-08-02 00:00:00
Validation from 1953-07-31 to 195

KeyboardInterrupt: 